# Stage 2 — Basic Data Preparation

**Project:** ResearchMate — Research Paper RAG Chatbot
**Goal of this notebook:** Prepare `train.csv` for RAG — inspect and remove any near-empty abstracts, combine TITLE + ABSTRACT into one text field, and build a readable `topics` metadata field. Save the result as `train_prepared.csv` for the next stage.

**Before running:** upload `train.csv` to this Colab session (same file used in Stage 1).

## Cell 1 — Load the dataset and detect columns

Same loading logic as Stage 1: read the CSV and auto-detect the ID/TITLE/ABSTRACT column names, so this notebook works even if column casing changes.

In [2]:
import pandas as pd

df = pd.read_csv("train.csv")

def find_col(possible_names, columns):
    for name in possible_names:
        for col in columns:
            if col.strip().lower() == name:
                return col
    return None

id_col = find_col(["id"], df.columns)
title_col = find_col(["title"], df.columns)
abstract_col = find_col(["abstract"], df.columns)

print("Shape:", df.shape)
print("ID column:", id_col, "| TITLE column:", title_col, "| ABSTRACT column:", abstract_col)

Shape: (20972, 9)
ID column: ID | TITLE column: TITLE | ABSTRACT column: ABSTRACT


## Cell 2 — Inspect very short abstracts

Stage 1 showed a minimum abstract length of 1 word (mean was 148 words). A near-empty abstract has essentially no semantic content and would just add noise to our vector store later. Before removing anything, we print every row with 5 words or fewer so we can see exactly what we're about to drop, instead of blindly filtering.

In [3]:
df["abstract_word_count"] = df[abstract_col].apply(lambda x: len(str(x).split()))

SHORT_THRESHOLD = 5  # words

short_rows = df[df["abstract_word_count"] <= SHORT_THRESHOLD]
print(f"Rows with abstract <= {SHORT_THRESHOLD} words: {len(short_rows)}")
short_rows[[id_col, title_col, abstract_col, "abstract_word_count"]]

Rows with abstract <= 5 words: 1


,ID,TITLE,ABSTRACT,abstract_word_count
16394,16395,Are theoretical results 'Results'?,Yes.\n,1


## Cell 3 — Drop the near-empty abstracts

Based on Cell 2's output, we remove rows at or below the short threshold — these abstracts don't carry enough information to be meaningfully embedded or retrieved. We print how many rows were removed and the new shape, so the change is visible and traceable.

In [4]:
before = len(df)

df = df[df["abstract_word_count"] > SHORT_THRESHOLD].reset_index(drop=True)

after = len(df)
print(f"Dropped {before - after} row(s) with near-empty abstracts.")
print("New shape:", df.shape)

Dropped 1 row(s) with near-empty abstracts.
New shape: (20971, 10)


## Cell 4 — Why we do NOT apply stemming, lemmatization, or stopword removal

This is a deliberate design choice, not an oversight:

- The embedding model we'll use in Stage 5 is trained on natural, fluent English sentences — not stemmed fragments like "predict subject-specif inform".
- Stopwords ("of", "the", "is") often carry grammatical relationships that help the embedding model understand meaning — removing them can actually *reduce* retrieval quality.
- Classic NLP preprocessing (stemming/lemmatization/stopword removal) was designed for older bag-of-words/TF-IDF pipelines, not for modern semantic embeddings.

So beyond removing near-empty rows, we leave the text exactly as written.

## Cell 5 — Combine TITLE + ABSTRACT into one text field

This `text` column is the raw material that becomes `page_content` in Stage 3. We keep it simple: title, a blank line, then the abstract.

In [5]:
df["text"] = df[title_col].astype(str) + "\n\n" + df[abstract_col].astype(str)

df[["text"]].head(3)

,text
0,Reconstructing Subject-Specific Effect Maps\n\...
1,Rotation Invariance Neural Network\n\n Rotati...
2,Spherical polyharmonics and Poisson kernels fo...


## Cell 6 — Build a readable `topics` metadata field

The dataset has 6 separate 0/1 topic columns. We collapse them into a single readable string per paper (e.g. `"Computer Science, Statistics"`), which will be much easier to display in our chatbot's UI later than 6 separate binary columns.

In [6]:
known_non_topic_cols = {id_col, title_col, abstract_col, "abstract_word_count", "text"}
topic_cols = [c for c in df.columns if c not in known_non_topic_cols]

print("Topic columns:", topic_cols)

def get_topics(row):
    return ", ".join([t for t in topic_cols if row[t] == 1])

df["topics"] = df.apply(get_topics, axis=1)

df[[title_col, "topics"]].head(5)

Topic columns: ['Computer Science', 'Physics', 'Mathematics', 'Statistics', 'Quantitative Biology', 'Quantitative Finance']


,TITLE,topics
0,Reconstructing Subject-Specific Effect Maps,Computer Science
1,Rotation Invariance Neural Network,Computer Science
2,Spherical polyharmonics and Poisson kernels fo...,Mathematics
3,A finite element approximation for the stochas...,Mathematics
4,Comparative study of Discrete Wavelet Transfor...,"Computer Science, Statistics"


## Cell 7 — Test the prepared data

A quick sanity check: print a couple of full examples of the final `text` + `topics` fields, and confirm no row ended up with an empty `topics` string (every paper should have at least one topic, as confirmed in Stage 1).

In [7]:
print("Rows with no topic assigned:", (df["topics"] == "").sum())

for i in range(2):
    print("=" * 60)
    print("TEXT:\n", df.iloc[i]["text"])
    print("\nTOPICS:", df.iloc[i]["topics"])

Rows with no topic assigned: 0
TEXT:
 Reconstructing Subject-Specific Effect Maps

  Predictive models allow subject-specific inference when analyzing disease
related alterations in neuroimaging data. Given a subject's data, inference can
be made at two levels: global, i.e. identifiying condition presence for the
subject, and local, i.e. detecting condition effect on each individual
measurement extracted from the subject's data. While global inference is widely
used, local inference, which can be used to form subject-specific effect maps,
is rarely used because existing models often yield noisy detections composed of
dispersed isolated islands. In this article, we propose a reconstruction
method, named RSM, to improve subject-specific detections of predictive
modeling approaches and in particular, binary classifiers. RSM specifically
aims to reduce noise due to sampling error associated with using a finite
sample of examples to train classifiers. The proposed method is a wrapper-type
a

## Cell 8 — Save the prepared dataset

We save the cleaned, prepared DataFrame to `train_prepared.csv`. The next stage (Document creation) will load this file directly instead of repeating Stage 1 + Stage 2.

In [8]:
output_cols = [id_col, title_col, abstract_col, "text", "topics"]
df[output_cols].to_csv("train_prepared.csv", index=False)

print("Saved train_prepared.csv with shape:", df[output_cols].shape)

Saved train_prepared.csv with shape: (20971, 5)


## What to check after running this notebook

- **Cell 2:** how many near-empty abstracts were found, and whether they look like genuinely bad data (they should).
- **Cell 3:** confirm the row count dropped only slightly (a handful of rows, not thousands) and the new shape looks right.
- **Cell 6:** spot-check a few `topics` values — do they look sensible given the titles?
- **Cell 7:** confirm 0 rows have an empty `topics` string, and that the printed `text` examples read naturally (title + blank line + abstract).
- **Cell 8:** confirm `train_prepared.csv` was created — download it from the Colab file browser (left sidebar) since we'll need to re-upload it in the Stage 3 notebook.

Paste back the row-drop count from Cell 3, and the two example outputs from Cell 7, so we can confirm everything looks right before moving to Document creation.